# Data

Data source:  `data/processed/clean/`

In [10]:
import numpy as np
import pandas as pd

## Data splits info

In [16]:
from attention_lapse_detection.utils.constants import WINDOW_SECONDS_OPTIONS
from attention_lapse_detection.utils.data_paths import clean_dir
from attention_lapse_detection.training.data import load_clip_ids
from attention_lapse_detection.training.metrics import participant_ids

SPLITS = ("Train", "Validation", "Test")

labels, clips = {}, {}
rows = []

for window in WINDOW_SECONDS_OPTIONS:
    for split in SPLITS:
        y = np.load(
            clean_dir(10, window_seconds=window) / f"y_{split}_windows_clean.npy"
        )
        ids = load_clip_ids(10, [], split, window)

        labels[window, split] = y
        clips[window, split] = ids

        clip_y = pd.Series(y == 0).groupby(ids).first()

        rows.append(
            {
                "window_s": window,
                "split": split,
                "participants": len(np.unique(participant_ids(ids))),
                "participants_with_disengaged": len(
                    np.unique(participant_ids(ids[y == 0]))
                ),
                "windows": len(y),
                "clips": len(clip_y),
                "disengaged_clips": int(clip_y.sum()),
                "engaged_clips": int((~clip_y).sum()),
                "prevalence": round(clip_y.mean(), 4),
            }
        )

samples = pd.DataFrame(rows).set_index(["window_s", "split"])
samples

participants  participants_with_disengaged  windows  \
window_s split                                                             
5        Train                 69                            56    10716   
         Validation            19                            18     2858   
         Test                  21                            16     3537   
10       Train                 69                            56     5358   
         Validation            19                            18     1429   
         Test                  21                            16     1753   

                     clips  disengaged_clips  engaged_clips  prevalence  
window_s split                                                           
5        Train        5358               247           5111      0.0461  
         Validation   1429               166           1263      0.1162  
         Test         1784                88           1696      0.0493  
10       Train        5358               247           5111      0.0461  
         Validation   1429               166           1263      0.1162  
         Test         1753                84           1669      0.0479

## Further infos

raw feature row alongside cleaned row.

In [25]:
from attention_lapse_detection.utils.data_paths import merged_npz
from attention_lapse_detection.utils.data_paths import load_clean_split
from attention_lapse_detection.utils.constants import FEATURE_COLUMNS

raw = np.load(merged_npz(10, "Train", window_seconds=10), allow_pickle=True)
X_train, _ = load_clean_split(10, [], "Train", 10)
first_clip = load_clip_ids(10, [], "Train", 10)[0]


def to_df(windows):
    return pd.DataFrame(
        windows.reshape(-1, len(FEATURE_COLUMNS)), columns=FEATURE_COLUMNS
    )


raw_frames = to_df(raw["X"])
clean_frames = to_df(X_train)

print(f"First frame of {first_clip}")

pd.DataFrame(
    {"raw": raw["X"][raw["video_id"] == first_clip][0, 0], "cleaned": X_train[0, 0]},
    index=FEATURE_COLUMNS,
).round(3)

First frame of 1100011002.avi


,raw,cleaned
ear,0.352,-0.095
mar,0.539,1.189
roll,2.160,0.647
pitch,8.140,1.844
yaw,1.420,0.383
perclos,0.000,-0.414
blinks,0.000,0.000
face_present,1.000,1.000
gaze_x,-0.001,-0.473
gaze_y,-0.002,-0.753


In [30]:
raw_frames.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
ear,535800.0,0.358,0.070,0.00,0.331,0.371,0.404,1.797
mar,535800.0,0.432,0.091,0.00,0.374,0.429,0.482,1.847
roll,535800.0,-1.490,5.874,-68.48,-5.060,-1.250,2.130,52.600
pitch,535800.0,-6.012,7.698,-37.06,-10.700,-5.760,-1.170,41.040
yaw,535800.0,-0.286,4.519,-48.91,-2.870,-0.290,2.510,68.250
perclos,535800.0,0.040,0.097,0.00,0.000,0.000,0.045,1.000
blinks,535800.0,0.019,0.137,0.00,0.000,0.000,0.000,1.000
face_present,535800.0,1.000,0.013,0.00,1.000,1.000,1.000,1.000
gaze_x,535800.0,-0.000,0.001,-0.02,-0.001,-0.000,0.001,0.014
gaze_y,535800.0,-0.001,0.001,-0.01,-0.002,-0.001,-0.000,0.032


In [31]:
clean_frames.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
ear,535800.0,-0.000,1.000,-4.497,-0.388,0.174,0.657,2.120
mar,535800.0,0.000,1.000,-3.062,-0.642,-0.026,0.559,4.353
roll,535800.0,0.002,0.986,-5.000,-0.636,0.039,0.641,3.482
pitch,535800.0,0.000,1.000,-3.105,-0.610,0.033,0.631,3.082
yaw,535800.0,0.000,1.000,-4.498,-0.579,-0.001,0.627,3.689
perclos,535800.0,-0.028,0.775,-0.414,-0.414,-0.414,0.056,5.000
blinks,535800.0,0.019,0.137,0.000,0.000,0.000,0.000,1.000
face_present,535800.0,1.000,0.013,0.000,1.000,1.000,1.000,1.000
gaze_x,535800.0,0.002,0.987,-5.000,-0.473,0.012,0.496,5.000
gaze_y,535800.0,-0.000,1.000,-3.615,-0.606,-0.092,0.642,4.458
